In [ ]:
# Run the visualization script
import subprocess
import os
from pathlib import Path

# Create output directory
outdir = 'figs/zinc12k_evaluation_MLP'
os.makedirs(outdir, exist_ok=True)

# Run visualization
cmd = [
    'python', 'visualize_latent_embeddings.py',
    '--embedding', '../scripts/final_logs/ZINC12K_regress_nokld_2026-01-07-14-45-42/ordered_embedding_ZINC12K_regress_nokld.npy',
    '--labels', '../scripts/final_logs/ZINC12K_regress_nokld_2026-01-07-14-45-42/embedding_prop_lists_ZINC12K_regress_nokld.npy',
    '--outdir', outdir,
    '--no-show'
]

print("Running visualization script (α=0.3, 16 node features)...")
print(f"Command: {' '.join(cmd)}")
print()

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stdout)
if result.stderr:
    print("Errors/Warnings:")
    print(result.stderr)

if result.returncode == 0:
    print("\n✅ Visualization complete!")
    print(f"Figures saved to: {outdir}")
    
    fig_files = list(Path(outdir).glob('*.png'))
    if fig_files:
        print("\nGenerated figures:")
        for f in sorted(fig_files):
            print(f"  - {f.name}")
else:
    print(f"\n❌ Visualization failed with return code {result.returncode}")


Running visualization script (α=0.3, 16 node features)...
Command: python visualize_latent_embeddings.py --embedding scripts/final_logs/ZINC12K_regress_nokld_2026-01-07-14-45-42/ordered_embedding_ZINC12K_regress_nokld.npy --labels scripts/final_logs/ZINC12K_regress_nokld_2026-01-07-14-45-42/embedding_prop_lists_ZINC12K_regress_nokld.npy --outdir figs/zinc12k_evaluation_MLP --no-show



Loading embedding: scripts/final_logs/ZINC12K_regress_nokld_2026-01-07-14-45-42/ordered_embedding_ZINC12K_regress_nokld.npy

Errors/Warnings:
Traceback (most recent call last):
  File "/nfs/roberts/project/pi_sk2433/jcr222/workspace/GRASSY-Net/notebooks/visualize_latent_embeddings.py", line 126, in <module>
    main()
  File "/nfs/roberts/project/pi_sk2433/jcr222/workspace/GRASSY-Net/notebooks/visualize_latent_embeddings.py", line 68, in main
    X = load_embedding(args.embedding)
  File "/nfs/roberts/project/pi_sk2433/jcr222/workspace/GRASSY-Net/notebooks/visualize_latent_embeddings.py", line 24, in load_embedding
    x = np.load(path)
  File "/home/jcr222/workspace/GRASSY-Net/.venv/lib/python3.10/site-packages/numpy/lib/_npyio_impl.py", line 451, in load
    fid = stack.enter_context(open(os.fspath(file), "rb"))
FileNotFoundError: [Errno 2] No such file or directory: 'scripts/final_logs/ZINC12K_regress_nokld_2026-01-07-14-45-42/ordered_embedding_ZINC12K_regress_nokld.npy'


❌ Visuali

In [3]:
# Display the generated figures
from IPython.display import Image, display
import glob
from pathlib import Path

outdir = 'figs/zinc12k_evaluation_alpha_03'  # Match the outdir from visualization script
fig_files = sorted(glob.glob(f'{outdir}/*.png'))
if fig_files:
    print(f"Found {len(fig_files)} figure(s):\n")
    for fig_path in fig_files:
        print(f"📊 {Path(fig_path).name}")
        display(Image(fig_path))
        print()
else:
    print(f"No figures found in {outdir}")


No figures found in figs/zinc12k_evaluation_alpha_03


In [ ]:
## 3. Test Model Accuracy

import torch
import numpy as np
from models.GRASSY_model import GRASSY
from argparse import Namespace
from datasets.ZINCDataset import ZINCDataset, Scattering
from tqdm import tqdm

# Load dataset
dataset = ZINCDataset('datasets/ZINC12K.npy', prop_stat_dict='datasets/ZINC12K_stats.npy',
                      transform=Scattering(scatter_model_name='scripts/trained_models/ZINC12K.npy'))

# Load model
model_path = 'scripts/final_logs/ZINC12K_regress_nokld_2026-01-07-14-45-42/ZINC12K_regress_nokld_model.npy'
model_state = torch.load(model_path, map_location='cpu')
sample_x, sample_y = dataset[0]

# Property names (adjust to match your dataset)
prop_names = ['qed', 'HeavyAtomMolWt', 'MolWt', 'BalabanJ', 'BertzCT', 'Ipc', 
              'TPSA', 'NumHAcceptors', 'NumHDonors', 'RingCount', 'MolLogP', 'SAscore', 'FSP3']

hparams = Namespace(input_dim=len(sample_x), bottle_dim=25, hidden_dim=100, 
                    learning_rate=0.001, alpha=0.3, n_epochs=100,
                    len_epoch=None, batch_size=100, n_gpus=0, num_properties=len(sample_y))
model = GRASSY(hparams=hparams)
model.load_state_dict(model_state)
model.eval()

# Test set (last 1000 molecules: original test split)
test_indices = list(range(11000, len(dataset)))
print(f"Testing on {len(test_indices)} molecules (test set)")
print("=" * 70)

# Store all errors and true values per property
all_abs_errors = []
all_true_values = []

with torch.no_grad():
    for idx in tqdm(test_indices, desc="Evaluating"):
        x, y = dataset[idx]
        x_t = x.unsqueeze(0).float()
        y_t = torch.tensor(y).unsqueeze(0).float()
        
        _, y_hat, _, _, _ = model(x_t)
        
        # Absolute error per property
        abs_error = torch.abs(y_hat - y_t).squeeze(0)
        all_abs_errors.append(abs_error.numpy())
        all_true_values.append(y_t.squeeze(0).numpy())

# Convert to arrays: [N_samples, N_properties]
all_abs_errors = np.array(all_abs_errors)
all_true_values = np.array(all_true_values)

print("\n" + "=" * 70)
print("PROPERTY PREDICTION ACCURACY (Per Property)")
print("=" * 70)
print(f"{'Property':<20} | {'MAE':<12} | {'Std Dev':<12} | {'Range (min-max)':<20}")
print("-" * 70)

# Calculate and print stats per property
for i in range(all_abs_errors.shape[1]):
    prop_mae = all_abs_errors[:, i].mean()
    prop_std = all_abs_errors[:, i].std()
    prop_min = all_true_values[:, i].min()
    prop_max = all_true_values[:, i].max()
    
    # Use property name if available
    p_name = prop_names[i] if i < len(prop_names) else f"Prop {i+1}"
    
    print(f"{p_name:<20} | {prop_mae:<12.6f} | {prop_std:<12.6f} | [{prop_min:.3f}, {prop_max:.3f}]")

print("=" * 70)
print("\nOverall (averaged across all properties):")
print(f"  Mean Absolute Error: {all_abs_errors.mean():.6f}")
print(f"  Standard Deviation: ±{all_abs_errors.std():.6f}")


ModuleNotFoundError: No module named 'models'

## Training Summary: α=0.3

### Configuration
- **Alpha (regression weight)**: 0.3
- **Node features**: 16 (8 atom types + 8 pair encodings)
- **Training directory**: `ZINC12K_regress_nokld_2026-01-05-15-15-11/`

